# Model 03 — KNN (Motor Failure Prediction)

This notebook provides a step-by-step analytical walkthrough of an incipient motor failure classification problem using K-Nearest Neighbors (KNN).

The motivation for this case is operationally simple: a reliability engineer wants an early warning signal that is actionable, not a perfect crystal ball. After reviewing the problem, KNN becomes a natural candidate because it learns risk based on similarity to historical operating conditions, capturing localized, non-linear patterns.

This notebook is designed to be executed sequentially. Each section explains its purpose before running the code.

Data source: `motor_failure_prediction_data.csv`
Output: metrics, decision map, feature importance, scenario simulator, baseline comparison, and a McNemar statistical test.


In [ ]:
# 1) Imports
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    average_precision_score
)

from sklearn.inspection import permutation_importance

# McNemar (approx. chi-square)
from scipy.stats import chi2

# Univariate logistic regression (statsmodels)
import statsmodels.api as sm

In [ ]:
# 2) Load dataset (CSV)

DATA_PATH = "motor_failure_prediction_data.csv"  # <-- change if needed
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
# 3) Quick sanity checks
df.info()

# Class balance
df["incipient_failure"].value_counts(dropna=False)

In [ ]:
# 4) Summary statistics
df.describe(include="all")

## 1. Exploratory Data Analysis (EDA)

We start with quick visual checks to confirm the data behaves like a realistic condition-monitoring scenario:
- distributions per feature
- boxplots by class (normal vs incipient failure)
- correlations
- a 2D scatter plot to preview separation in the feature space

In [ ]:
features = ["vib_rms", "vib_pp", "temp_bearing", "current", "freq_dom", "load_pct"]
target = "incipient_failure"

In [ ]:
# 1.1 Histograms
df[features].hist(figsize=(12, 8), bins=30)
plt.tight_layout()
plt.show()

In [ ]:
# 1.2 Boxplots by class
plt.figure(figsize=(12, 8))
for i, col in enumerate(features, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(data=df, x=target, y=col)
    plt.title(col)
plt.tight_layout()
plt.show()

In [ ]:
# 1.3 Correlation heatmap
plt.figure(figsize=(6, 5))
corr = df[features + [target]].corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
# 1.4 Scatter: vib_rms vs temp_bearing
plt.figure(figsize=(6, 5))
sns.scatterplot(
    data=df,
    x="vib_rms",
    y="temp_bearing",
    hue=target,
    alpha=0.6
)
plt.title("vib_rms vs temp_bearing by class")
plt.tight_layout()
plt.show()

## 2. KNN Model Training (Hyperparameter Tuning)

KNN is distance-based, so **feature scaling** is required.  
We tune:
- number of neighbors (`n_neighbors`)
- distance metric (`p`: Manhattan vs Euclidean)
- weighting (`uniform` vs `distance`)

In [ ]:
# 2.1 Train/test split
X = df[features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

In [ ]:
# 2.2 Pipeline + GridSearchCV (KNN)
pipe_knn = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier())
])

param_grid_knn = {
    "knn__n_neighbors": [3, 5, 7, 9, 11, 15],
    "knn__weights": ["uniform", "distance"],
    "knn__p": [1, 2]  # 1 = Manhattan, 2 = Euclidean
}

grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid_knn,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("Best KNN params:", grid_knn.best_params_)
print("Best CV accuracy:", grid_knn.best_score_)
best_knn = grid_knn.best_estimator_

## 3. Model Evaluation

We report:
- Accuracy, ROC-AUC
- Classification report (precision/recall/F1)
- Confusion matrix (absolute + normalized)
- Precision–Recall curve

In [ ]:
# 3.1 Predictions + core metrics
y_pred_knn = best_knn.predict(X_test)
y_proba_knn = best_knn.predict_proba(X_test)[:, 1]

print("KNN — Accuracy (test):", accuracy_score(y_test, y_pred_knn))
print("KNN — ROC-AUC (test):", roc_auc_score(y_test, y_proba_knn))
print("\nClassification report (KNN):\n")
print(classification_report(y_test, y_pred_knn))

In [ ]:
# 3.2 Confusion matrices (absolute + normalized)
cm = confusion_matrix(y_test, y_pred_knn)
cm_norm = confusion_matrix(y_test, y_pred_knn, normalize="true")

labels = ["Normal (0)", "Incipient failure (1)"]

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion matrix (absolute)")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.subplot(1, 2, 2)
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title("Confusion matrix (normalized)")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# 3.3 Precision–Recall curve
precision, recall, _ = precision_recall_curve(y_test, y_proba_knn)
ap = average_precision_score(y_test, y_proba_knn)

plt.figure(figsize=(6, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(f"Precision–Recall (AP = {ap:.3f})")
plt.grid(True)
plt.tight_layout()
plt.show()

## 4. 2D Decision Map (vib_rms vs temp_bearing)

To visualize how KNN separates regions, we project the decision surface on:
- **x:** `vib_rms`
- **y:** `temp_bearing`

All other variables are fixed to their mean (from the training set).

In [ ]:
# 4.1 Build a grid in the (vib_rms, temp_bearing) plane
vib_min, vib_max = X["vib_rms"].min() - 0.5, X["vib_rms"].max() + 0.5
temp_min, temp_max = X["temp_bearing"].min() - 2, X["temp_bearing"].max() + 2

xx, yy = np.meshgrid(
    np.linspace(vib_min, vib_max, 200),
    np.linspace(temp_min, temp_max, 200)
)

means = X_train.mean(numeric_only=True)

grid_points = []
for vx, tx in zip(xx.ravel(), yy.ravel()):
    grid_points.append({
        "vib_rms": vx,
        "temp_bearing": tx,
        "vib_pp": means["vib_pp"],
        "current": means["current"],
        "freq_dom": means["freq_dom"],
        "load_pct": means["load_pct"]
    })

grid_df = pd.DataFrame(grid_points, columns=features)

proba_grid = best_knn.predict_proba(grid_df)[:, 1].reshape(xx.shape)

In [ ]:
# 4.2 Plot decision map + test points
plt.figure(figsize=(6, 5))

contour = plt.contourf(
    xx, yy, proba_grid,
    levels=20,
    cmap="RdYlBu",
    alpha=0.7
)
plt.colorbar(contour, label="Predicted probability of failure")

plt.scatter(
    X_test["vib_rms"],
    X_test["temp_bearing"],
    c=y_test,
    cmap="bwr",
    edgecolor="k",
    alpha=0.8
)

plt.xlabel("vib_rms")
plt.ylabel("temp_bearing")
plt.title("KNN decision map (vib_rms vs temp_bearing)")
plt.tight_layout()
plt.show()

## 5. Feature Importance (Permutation Importance)

KNN has no coefficients. To estimate feature influence, we use **permutation importance**:
- shuffle a feature
- measure how much the score drops

In [ ]:
# 5.1 Permutation importance on the test set
perm = permutation_importance(
    best_knn, X_test, y_test,
    n_repeats=10,
    random_state=42,
    n_jobs=-1
)

importances = pd.DataFrame({
    "feature": features,
    "importance_mean": perm.importances_mean,
    "importance_std": perm.importances_std
}).sort_values("importance_mean", ascending=False)

importances

In [ ]:
# 5.2 Pareto-style plot (top 10)
top_n = 10
top_imp = importances.head(top_n)

plt.figure(figsize=(6, 5))
sns.barplot(data=top_imp, x="importance_mean", y="feature")
plt.xlabel("Mean decrease in score when permuted")
plt.ylabel("Feature")
plt.title("Permutation importance (KNN) — Top features")
plt.tight_layout()
plt.show()

## 6. Simple Simulator (What-if Scenarios)

This helper function lets you enter a hypothetical motor condition and get:
- predicted class (0/1)
- predicted probability of incipient failure

In [ ]:
def simulate_motor(vib_rms, vib_pp, temp_bearing, current, freq_dom, load_pct):
    row = pd.DataFrame([{
        "vib_rms": vib_rms,
        "vib_pp": vib_pp,
        "temp_bearing": temp_bearing,
        "current": current,
        "freq_dom": freq_dom,
        "load_pct": load_pct
    }], columns=features)

    proba = best_knn.predict_proba(row)[:, 1][0]
    pred = int(best_knn.predict(row)[0])
    return proba, pred

# Example: "healthy" motor
p_healthy, y_healthy = simulate_motor(
    vib_rms=2.0, vib_pp=5.0, temp_bearing=60,
    current=28, freq_dom=30, load_pct=70
)
print("Healthy example -> proba:", round(p_healthy, 3), "pred:", y_healthy)

# Example: "at-risk" motor
p_risk, y_risk = simulate_motor(
    vib_rms=4.8, vib_pp=9.8, temp_bearing=88,
    current=37, freq_dom=23, load_pct=92
)
print("At-risk example -> proba:", round(p_risk, 3), "pred:", y_risk)

## 7. Baseline Comparison and McNemar Test

We compare KNN against:
- Logistic Regression (linear baseline)
- Naive Bayes (probabilistic baseline)

Then we run **McNemar** between KNN and Logistic Regression to verify whether the difference is statistically meaningful.

In [ ]:
# 7.1 Train baselines

# Logistic Regression
pipe_logit = Pipeline([
    ("scaler", StandardScaler()),
    ("logit", LogisticRegression(max_iter=1000))
])
pipe_logit.fit(X_train, y_train)
y_pred_logit = pipe_logit.predict(X_test)
y_proba_logit = pipe_logit.predict_proba(X_test)[:, 1]

# Naive Bayes (Gaussian)
pipe_nb = Pipeline([
    ("scaler", StandardScaler()),
    ("nb", GaussianNB())
])
pipe_nb.fit(X_train, y_train)
y_pred_nb = pipe_nb.predict(X_test)
y_proba_nb = pipe_nb.predict_proba(X_test)[:, 1]

In [ ]:
# 7.2 Metrics table (same split)
def model_metrics(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    auc = roc_auc_score(y_true, y_proba)
    rep = classification_report(y_true, y_pred, output_dict=True)
    f1_pos = rep["1"]["f1-score"]  # failure class
    return acc, auc, f1_pos

acc_knn, auc_knn, f1_knn = model_metrics(y_test, y_pred_knn, y_proba_knn)
acc_log, auc_log, f1_log = model_metrics(y_test, y_pred_logit, y_proba_logit)
acc_nb,  auc_nb,  f1_nb  = model_metrics(y_test, y_pred_nb,  y_proba_nb)

results = pd.DataFrame({
    "Model": ["KNN", "Logistic Regression", "Naive Bayes"],
    "Accuracy": [acc_knn, acc_log, acc_nb],
    "ROC-AUC": [auc_knn, auc_log, auc_nb],
    "F1 (failure=1)": [f1_knn, f1_log, f1_nb]
})

results

In [ ]:
# 7.3 McNemar test (KNN vs Logistic Regression)
correct_knn = (y_pred_knn.values if hasattr(y_pred_knn, "values") else y_pred_knn) == y_test.values
correct_log = (y_pred_logit.values if hasattr(y_pred_logit, "values") else y_pred_logit) == y_test.values

n01 = np.sum((correct_knn == False) & (correct_log == True))   # KNN wrong, Logit right
n10 = np.sum((correct_knn == True) & (correct_log == False))   # KNN right, Logit wrong

print("n01 (KNN wrong, Logit right):", n01)
print("n10 (KNN right, Logit wrong):", n10)

if n01 + n10 == 0:
    print("No disagreements between models. McNemar test is not applicable.")
else:
    chi2_stat = (abs(n01 - n10) - 1)**2 / (n01 + n10)  # continuity correction
    p_value = 1 - chi2.cdf(chi2_stat, df=1)

    print(f"McNemar chi2: {chi2_stat:.4f}")
    print(f"p-value: {p_value:.4f}")

## 8. Univariate Logistic Regression (One Variable at a Time)

To complement KNN (non-parametric), we fit **one logistic regression per feature** to get:
- coefficient sign (direction of effect)
- odds ratio + confidence interval
- p-value (significance)

This is **interpretability**, not necessarily the best predictive performance.

In [ ]:
variables = features.copy()
y_uni = df[target]

models_uni = {}

for var in variables:
    X_uni = sm.add_constant(df[[var]])
    m = sm.Logit(y_uni, X_uni).fit(disp=False)
    models_uni[var] = m
    print("\n" + "="*60)
    print(f"Univariate Logistic Regression: {var} -> incipient_failure")
    print("="*60)
    print(m.summary())

In [ ]:
# Odds ratios (with 95% CI)
print("\nODDS RATIO SUMMARY (per variable)\n")

rows = []
for var, m in models_uni.items():
    params = m.params
    conf = m.conf_int()
    beta1 = float(params[var])
    pval = float(m.pvalues[var])

    or_ = float(np.exp(beta1))
    ci_low = float(np.exp(conf.loc[var, 0]))
    ci_high = float(np.exp(conf.loc[var, 1]))

    rows.append([var, beta1, or_, ci_low, ci_high, pval])

or_table = pd.DataFrame(rows, columns=["Variable", "Beta1", "OddsRatio", "CI95_low", "CI95_high", "p_value"])
or_table.sort_values("OddsRatio", ascending=False)

In [ ]:
# Probability curves (one plot per variable)
for var in variables:
    X_plot = df[[var]].values.reshape(-1, 1)
    y_plot = df[target].values

    lr = LogisticRegression(max_iter=1000).fit(X_plot, y_plot)

    x_min, x_max = df[var].min(), df[var].max()
    x_range = np.linspace(x_min, x_max, 400).reshape(-1, 1)
    y_prob = lr.predict_proba(x_range)[:, 1]

    plt.figure(figsize=(6, 4))
    plt.scatter(df[var], df[target], alpha=0.2, label="Data")
    plt.plot(x_range, y_prob, linewidth=2, label="Logistic curve")
    plt.xlabel(var)
    plt.ylabel("P(incipient_failure=1)")
    plt.title(f"Univariate logistic: {var} -> failure probability")
    plt.legend()
    plt.tight_layout()
    plt.show()

## Technical Takeaways (Short)

- KNN is effective here because failure risk emerges in **localized, non-linear regions** of the operating space.
- Permutation importance typically highlights vibration RMS and bearing temperature as dominant drivers.
- Baseline comparison + McNemar provides evidence that KNN improves over a linear baseline for this dataset.
- Univariate logistic models help quantify directionality via odds ratios (interpretability layer).

—
LozanoLsa
Regards from MX